# 검출 fine-tuning (TASK 6-A / M3)

목표: 시스템 Top-5 **62.9% → ≥81%**

근거 문서 — `docs/metrics_policy.md`, `reports/m2b_spatial_group.md`

## 왜 학습인가
저비용 경로는 전부 소진됐다: 후처리 스윕 개선 0%(M1), GT 없는 조각 재결합 +1.0%p(M2-B).
남은 원인은 **박스 규약 불일치**다 — `8196 P32`는 한 박스인데 옆의 `P32`는 다른 박스이고,
이 구분은 기하 규칙으로 복원할 수 없다. 그러나 **학습은 가능하다.**

## 게이트
- **셀 4** Phase 0 스모크 테스트를 GPU 에서 재실행. 실패하면 여기서 중단(로컬은 CPU 로만 검증했다).
- **셀 6** PP-OCRv4 det baseline 재측정. 지금까지 측정은 PP-OCRv5_server_det(3.x 전용)인데
  학습은 2.x(최대 v4)라, **같은 계열로 기준선을 다시 잡지 않으면 학습 효과와 세대 차가 섞인다.**

## 1. 환경 — GPU 확인, 버전 고정, 검증

설치·복구·검증을 한 셀에 모았다. **pip 의 "dependency conflicts" 경고는 정상**이며,
판정은 셀 끝의 import 검증이 통과하는지로 한다.
이 셀은 `sh()` 도 정의하므로 **세션이 초기화되면 반드시 먼저 실행해야 한다.**

In [ ]:
!nvidia-smi

# paddlepaddle-gpu 는 PyPI 에 없다. Baidu 공식 인덱스를 지정해야 한다.
# CUDA 버전은 위 nvidia-smi 출력에 맞춰 cu126 / cu118 등으로 교체할 것.
!python -m pip install -q "paddlepaddle-gpu==3.3.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install -q "paddleocr==3.7.0" rapidfuzz shapely lmdb

# [필수] 위 설치가 Colab 기본 torch 의 nvidia-* 핀을 덮어써서 torch 가 깨진다:
#   ImportError: libtorch_cuda.so: undefined symbol: ncclCommShrink
# 우리는 torch 를 쓰지 않지만 전이 import 로 반드시 끌려온다:
#   src/preprocess/crop.py -> paddlex -> official_models.py 의 `import modelscope`
#   (가드 없는 최상위 import) -> modelscope 로거 -> `import torch`
# torch 가 요구하는 nccl 로 되돌린다. 단일 GPU 학습에서 paddle 은 nccl 을 쓰지 않으므로
# (nccl 은 다중 카드 collective 통신용) paddle 쪽은 영향받지 않는다.
!python -m pip install -q nvidia-nccl-cu12==2.28.9

import subprocess


def sh(cmd, cwd=None):
    """서브프로세스를 돌리고 출력을 **셀에 보이게** 흘려보낸 뒤 종료 코드를 반환한다.

    subprocess 는 OS 레벨 fd 1 에 쓰는데 노트북 프런트엔드는 파이썬 레벨 sys.stdout 만
    가져간다. 그래서 capture 없이 돌리면 출력이 셀에 나타나지 않고 서버 로그로 샌다
    (`!` 매직이 보이는 건 IPython 이 파이프로 받아 되뿌리기 때문이다).
    여기서는 파이프로 받아 print 로 되뿌린다. 자식 쪽 버퍼링은 `-u` 로 끈다 —
    끄지 않으면 긴 작업의 진행 상황이 끝날 때 한꺼번에 쏟아진다.
    """
    if cmd[0] == "python":
        cmd = [cmd[0], "-u", *cmd[1:]]
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()


# 위 pip 이 "dependency conflicts" 를 몇 줄 뱉는 것은 **정상이며 무시한다.**
# pip 은 설치된 메타데이터의 핀만 보고 짖을 뿐 실제로 로드되는지는 모른다.
# 판정 기준은 pip 이 조용한가가 아니라 아래가 통과하는가다.
print("=" * 64)
assert sh(["python", "-c",
           "import torch, paddle, paddlex;"
           "assert paddle.device.is_compiled_with_cuda(),"
           " 'GPU 빌드가 아니다 — 런타임 유형을 GPU 로 바꿀 것';"
           "print('torch', torch.__version__, '| paddle', paddle.__version__,"
           "      '| devices', paddle.device.cuda.device_count())"]) == 0, \
    "환경 검증 실패 — 위 출력 확인"

## 2. 데이터 — Drive 에서 세션 로컬 디스크로 해제

**Drive 에서 이미지를 직접 읽으면 DataLoader 가 심하게 느려진다.** 이미지는 로컬 디스크(`/content`)에,
체크포인트만 Drive 에 둔다(지시서 Colab 규약).

사전 준비(로컬 PC 에서 1회): `tar -czf label_ocr_images.tar.gz image_set` 후 Drive 에 업로드.
(Windows 의 bsdtar 는 후행 슬래시 `image_set/` 를 붙이면 인자 파싱이 깨진다.)

In [ ]:
import os
from google.colab import drive

DRIVE = '/content/drive/MyDrive/label_ocr'
ROOT  = '/content/label_ocr'          # 세션 로컬 (빠름)

# `MessageError: credential propagation was unsuccessful` 는 코드 결함이 아니라
# 브라우저에서 권한 팝업이 막혔거나 세션 자격증명이 전파되지 않은 것이다.
# 대개 재시도로 풀리므로 세 번 시도하고, 그래도 안 되면 조치 방법을 안내한다.
if not os.path.isdir('/content/drive/MyDrive'):
    for attempt in range(1, 4):
        try:
            drive.mount('/content/drive', force_remount=attempt > 1)
            break
        except Exception as e:
            print(f"마운트 실패 {attempt}/3: {type(e).__name__}: {e}")
    else:
        raise RuntimeError(
            "Drive 마운트 실패 — 코드로 풀 수 없는 브라우저·계정 문제다. 순서대로 시도할 것:\n"
            "  1) 좌측 파일 패널의 '드라이브 마운트' 버튼으로 직접 마운트\n"
            "  2) 팝업·서드파티 쿠키 차단 해제 (colab.research.google.com 허용)\n"
            "  3) 시크릿 창이면 일반 창으로. 구글 계정이 여럿이면 해당 계정만 로그인\n"
            "  4) 런타임 > 세션 관리 에서 세션 종료 후 재연결")

os.makedirs(f"{DRIVE}/experiments", exist_ok=True)
os.makedirs(ROOT, exist_ok=True)

# 이미지 해제 (약 1.4GB). 이미 풀려 있으면 건너뛴다 — 재실행이 몇 분씩 걸릴 이유가 없다.
if not os.path.isdir(f"{ROOT}/image_set"):
    assert sh(["tar", "-xzf", f"{DRIVE}/label_ocr_images.tar.gz", "-C", ROOT]) == 0, \
        "이미지 해제 실패 — Drive 에 label_ocr_images.tar.gz 가 있는지 확인"

n = sum(1 for f in os.listdir(f"{ROOT}/image_set") if f.lower().endswith(".jpg"))
print(f"이미지 {n}장")
assert n == 2340, f"이미지 수가 다르다 (기대 2340, 실제 {n}) — 아카이브를 다시 만들 것"

## 2-T. (선택) 원격 제어 터널 — **유료 플랜 전용**

Colab 런타임 안에 `sshd` 를 띄우고 cloudflared 임시 터널로 노출한다.
그러면 로컬 PC 에서 셸로 붙어 셀 3~8 을 직접 몰 수 있다 — 로그를 사람이 복사해 나르는
왕복이 사라진다.

> **무료 티어에서는 실행하지 말 것.** Colab FAQ 는 무료 사용자에 대해
> `remote control such as SSH shells, remote desktops` 를 명시적으로 금지하며,
> 이 제한은 유료 플랜 + 컴퓨팅 단위 잔액이 있을 때만 해제된다.

`PUBKEY` 는 **이미 채워져 있다.** 손대지 말 것 — 공개키는 비밀이 아니지만,
붙여넣다 줄바꿈이 문자열 안으로 들어가면 첫 줄에서 `SyntaxError` 로 죽는다(실제로 겪었다).

대응하는 개인키는 로컬 PC 의 `~/.ssh/colab_label_ocr` 에 있고 이 PC 를 떠나지 않는다.
비밀번호 인증은 꺼 두므로 이 키를 가진 쪽만 접속할 수 있다.
터널 주소는 실행할 때마다 새로 발급되는 임시 주소이고, 런타임이 죽으면 함께 사라진다.

키를 새로 만들어야 한다면:

```powershell
ssh-keygen -t ed25519 -N '""' -C "colab@label_ocr" -f "$env:USERPROFILE\.ssh\colab_label_ocr"
Get-Content "$env:USERPROFILE\.ssh\colab_label_ocr.pub"
```

In [ ]:
PUBKEY = ("ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIn7yySYlSTZ+LdBifePRnhTcwfOKFiOy7N0Di1f6ATX"
          " colab@label_ocr")

import os, re, socket, subprocess, textwrap, time

assert PUBKEY.startswith("ssh-"), "PUBKEY 를 먼저 채울 것"

# --- 1) sshd -------------------------------------------------------------
# 실패가 보이도록 sh() 로 돈다 (셀 1 에서 정의). subprocess 를 그냥 쓰면 출력이 셀에
# 나오지 않고 서버 로그로 새서, apt 가 왜 죽었는지 알 수 없게 된다.
assert sh(["apt-get", "-qq", "update"]) == 0, "apt update 실패"
assert sh(["apt-get", "-qq", "install", "-y", "openssh-server"]) == 0, "openssh-server 설치 실패"
assert sh(["ssh-keygen", "-A"]) == 0, "호스트 키 생성 실패"

# sshd 는 권한 분리용 /run/sshd 가 없으면 뜨지 않는다:
#   "Missing privilege separation directory: /run/sshd"
# 컨테이너 이미지에는 이 디렉터리가 없는 경우가 많다.
os.makedirs("/run/sshd", exist_ok=True)

# StrictModes 가 기본 on 이라 권한이 느슨하면 키 인증이 조용히 거부된다.
# makedirs 의 mode 는 '이미 존재하는' 디렉터리에는 적용되지 않으므로 따로 chmod 한다.
os.makedirs("/root/.ssh", exist_ok=True)
os.chmod("/root/.ssh", 0o700)
with open("/root/.ssh/authorized_keys", "w") as f:
    f.write(PUBKEY.strip() + "\n")
os.chmod("/root/.ssh/authorized_keys", 0o600)

# 배포판 기본 sshd_config 를 고치지 않는다. OpenSSH 는 같은 키워드의 '첫' 값을 채택하고
# Ubuntu 는 sshd_config.d 를 Include 하므로, 기본 파일 수정은 레이아웃에 따라 조용히
# 무시될 수 있다. 전용 config + -f 로 그 불확실성을 없앤다.
CONF = "/etc/ssh/sshd_colab.conf"
with open(CONF, "w") as f:
    f.write(textwrap.dedent("""\
        Port 22
        HostKey /etc/ssh/ssh_host_ed25519_key
        PermitRootLogin prohibit-password
        PubkeyAuthentication yes
        AuthorizedKeysFile /root/.ssh/authorized_keys
        PasswordAuthentication no
        KbdInteractiveAuthentication no
        UsePAM no
        ClientAliveInterval 60
        Subsystem sftp /usr/lib/openssh/sftp-server
    """))
sh(["pkill", "-f", "sshd -f"])      # 없으면 1 을 반환한다. 정상이므로 확인하지 않는다.
assert sh(["/usr/sbin/sshd", "-f", CONF]) == 0, "sshd 기동 실패"

# 정말 듣고 있는지 확인한다. 아니면 터널만 뚫리고 접속 단계에서야 정체불명으로 실패한다.
for _ in range(10):
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", 22)) == 0:
            break
    time.sleep(1)
else:
    raise RuntimeError("sshd 가 22번 포트에서 듣지 않는다")
print("sshd OK (22번 포트 응답)")

# --- 2) cloudflared 임시 터널 (계정·토큰 불필요) --------------------------
BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(BIN):
    # -f 필수. 없으면 404 응답 본문(HTML)을 파일에 써 놓고도 종료 코드 0 을 준다.
    assert sh(["curl", "-fsSL", "-o", BIN,
               "https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64"]) == 0, "cloudflared 다운로드 실패"
    os.chmod(BIN, 0o755)
assert sh([BIN, "--version"]) == 0, "cloudflared 바이너리가 실행되지 않는다"

# 주소는 로그로만 나온다. 로그 플래그 이름이 버전마다 달라 리다이렉트로 받는다.
LOG = "/content/cloudflared.log"
subprocess.Popen([BIN, "tunnel", "--no-autoupdate", "--url", "ssh://localhost:22"],
                 stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

host = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://([-\w]+\.trycloudflare\.com)", open(LOG).read())
    if m:
        host = m.group(1)
        break
if not host:
    print(open(LOG).read()[-3000:])
    raise RuntimeError("터널 주소를 못 받았다 — 위 로그 확인")

print("=" * 72)
print("터널 주소:", host)
print("=" * 72)

## 3. 코드 — 우리 저장소 + PaddleOCR 2.10.0

> **`ocngrn/label_ocr` 는 private 저장소다.** Colab 에는 자격증명이 없으므로 익명 clone 이
> `fatal: could not read Username for 'https://github.com'` 로 실패한다.
> 아래 셀이 실패를 감지하면 **PAT 입력창**을 띄운다 (`getpass` — 노트북에 남지 않는다).
> 매번 입력하기 싫으면 GitHub 에서 저장소를 public 으로 바꾸면 된다(이미지·라벨은 저장소에 없다).

In [ ]:
import os, shutil, subprocess, getpass

DEST = "/content/label_ocr_repo"
REPO = "github.com/ocngrn/label_ocr.git"

def run(*args, shell=False):
    """실패를 조용히 넘기지 않는다.
    이전 판은 `!git clone` 이 실패해도 다음 줄로 흘러가 copytree 에서 엉뚱한
    FileNotFoundError 로 드러났다. 진짜 원인(인증 실패)은 위쪽 출력에 묻혔다."""
    cmd = args[0] if shell else list(args)
    return subprocess.run(cmd, shell=shell, capture_output=True, text=True,
                          env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

# 저장소가 private 이라 Colab 에는 자격증명이 없다. 공개로 바뀌었을 수도 있으니 익명으로
# 먼저 시도하고, 실패하면 PAT 를 입력받는다. getpass 라 노트북 파일·출력에 남지 않는다.
if not os.path.isdir(DEST):
    r = run("git", "clone", "-q", f"https://{REPO}", DEST)
    if r.returncode != 0:
        print("익명 clone 실패 (private 저장소) — GitHub PAT 가 필요하다.")
        print("  발급: https://github.com/settings/tokens")
        print("  classic 이면 scope=repo, fine-grained 면 Contents=Read")
        tok = getpass.getpass("PAT: ").strip()
        r = run("git", "clone", "-q", f"https://{tok}@{REPO}", DEST)
        del tok
    assert r.returncode == 0, f"clone 실패:\n{r.stderr}"

# 이미지는 저장소에 없으므로 코드만 ROOT 로 합친다.
# 주의: IPython 의 `!` 는 {..} 를 파이썬 식으로 치환하므로 bash 중괄호 확장을 쓰지 않는다.
for name in ("src", "tests", "labels", "splits", "snapshots", "configs"):
    shutil.copytree(f"{DEST}/{name}", f"{ROOT}/{name}", dirs_exist_ok=True)
shutil.copy(f"{DEST}/conftest.py", ROOT)
print(sorted(os.listdir(ROOT)))

# 학습 진입점은 2.x 에만 있다 (3.7 pip 패키지는 추론 전용)
if not os.path.isdir("/content/PaddleOCR"):
    r = run("git", "clone", "-q", "--depth", "1", "--branch", "v2.10.0",
            "https://github.com/PaddlePaddle/PaddleOCR.git", "/content/PaddleOCR")
    assert r.returncode == 0, f"PaddleOCR clone 실패:\n{r.stderr}"

# 사전학습 가중치 (PP-OCRv4 server det)
W = "/content/weights/ch_PP-OCRv4_det_server_train"
if not os.path.exists(f"{W}/best_accuracy.pdparams"):
    os.makedirs("/content/weights", exist_ok=True)
    r = run("cd /content/weights"
            " && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/"
            "ch_PP-OCRv4_det_server_train.tar"
            " && tar xf ch_PP-OCRv4_det_server_train.tar", shell=True)
    assert r.returncode == 0, f"가중치 다운로드 실패:\n{r.stderr}"
assert os.path.exists(f"{W}/best_accuracy.pdparams"), "가중치 해제 실패"
print(sorted(os.listdir(W)))

## 4. [게이트] Phase 0 스모크 테스트 — GPU 재실행

`docs/framework_decision.md` 4장은 **CPU 에서만** 검증했다. GPU 빌드에서 2.10.0 코드가
도는지 여기서 확인한다. 실패하면 학습에 GPU 시간을 쓰지 않는다.

In [ ]:
import textwrap

# 서브프로세스로 돌린다. 2.10.0 을 쓰려면 sys.path 앞에 /content/PaddleOCR 을 꽂아야 하는데,
# 커널에 그게 남으면 이후 `import paddleocr` 가 pip 의 3.7.0 패키지 대신 2.x 저장소의
# paddleocr.py 를 집는다(-> ppstructure -> ModuleNotFoundError: docx). 셀 6 이 그래서 죽었다.
smoke = textwrap.dedent("""
    import sys, yaml, numpy as np, paddle
    sys.path.insert(0, '/content/PaddleOCR')
    from ppocr.modeling.architectures import build_model

    cfg = yaml.safe_load(open('/content/PaddleOCR/configs/det/ch_PP-OCRv4/'
                              'ch_PP-OCRv4_det_teacher.yml', encoding='utf-8'))
    model = build_model(cfg['Architecture'])

    sd = paddle.load('/content/weights/ch_PP-OCRv4_det_server_train/best_accuracy.pdparams')
    msd = model.state_dict()
    match = [k for k in msd if k in sd and tuple(msd[k].shape) == tuple(sd[k].shape)]
    print(f"파라미터 {len(msd)} 중 전이 {len(match)}, 미존재 {len([k for k in msd if k not in sd])}")
    model.set_state_dict({k: sd[k] for k in match})

    out = model(paddle.to_tensor(np.zeros((1, 3, 640, 640), 'float32')))
    print("GPU forward OK ->", {k: tuple(v.shape) for k, v in out.items()})
    assert len(match) / len(msd) > 0.9, "가중치 전이율이 낮다 — 템플릿/가중치 조합 확인"
""")
assert sh(["python", "-c", smoke]) == 0, "GPU 스모크 테스트 실패 — 여기서 중단한다"

## 5. 크롭 재생성 + 불변 규칙 테스트

크롭(233MB)은 전송하지 않고 여기서 만든다.
출력 수치가 로컬 `reports/data_profile_v1.md` 와 일치해야 학습 결과를 로컬 측정값과 비교할 수 있다.

In [ ]:
import os
os.chdir(ROOT)
os.makedirs("reports", exist_ok=True)

# `!` 는 종료 코드를 무시한다. 실제로 build_labels 가 죽었는데도 build_db 가 그대로 돌아
# 실패가 한 단계 늦게, 엉뚱한 모습으로 드러났다. 여기서는 단계마다 멈춘다.
for step in (["python", "-m", "src.preprocess.build_labels"],
             ["python", "-m", "src.matching.build_db"],
             ["python", "-m", "pytest", "tests/", "-q"]):
    print("$", " ".join(step), flush=True)
    assert sh(step) == 0, f"실패: {' '.join(step)}"

## 6. [게이트] PP-OCRv4 det baseline 재측정

**이 셀을 건너뛰면 학습 효과를 측정할 수 없다.** 기존 baseline 은 PP-OCRv5_server_det 로 쟀는데
학습은 v4 계열이므로, 같은 계열의 기준선을 여기서 확보한다.

In [ ]:
import os, json
os.chdir(ROOT)

# 서브프로세스로 돌려 깨끗한 sys.path 를 쓴다 (셀 4 주석 참조).
assert sh(["python", "-m", "src.eval.detect_baseline",
           "--model", "PP-OCRv4_server_det",     # 학습과 같은 계열로 기준선 확보
           "--out", "baseline_det_v4.json"]) == 0, "v4 baseline 측정 실패"

m = json.load(open("reports/baseline_det_v4.json", encoding="utf-8"))["PP-OCRv4_server_det"]
print(json.dumps({k: m[k] for k in ("overall", "plane", "curved")}, ensure_ascii=False, indent=2))
print("검출 recall(평면):", f"{m['plane']['recall']*100:.1f}%",
      "-> 시스템 Top-5 추정:", f"{m['plane']['recall']*0.856*100:.1f}%")

## 7. 학습

`save_epoch_step=5` + `checkpoints` 로 세션이 끊겨도 이어서 학습한다.
**세션이 끊기면 셀 1~3 을 다시 돌린 뒤 아래 `RESUME` 만 켜고 재실행하면 된다.**
(셀 1 은 `sh()` 정의, 셀 2 는 `ROOT`/`DRIVE` 정의라 건너뛸 수 없다.
셀 4·6 게이트는 한 번 통과했으면 생략해도 되지만, 셀 5 의 크롭은 세션 로컬 디스크에
있으므로 런타임이 초기화됐다면 다시 돌려야 한다.)

In [ ]:
import os
os.chdir(ROOT)
RESUME = None     # 이어서 학습할 때: f"{DRIVE}/experiments/det_v4_server/latest"

cmd = ["python", "-m", "src.configs.build_det_config",
       "--template", "/content/PaddleOCR/configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_teacher.yml",
       "--out", f"{ROOT}/configs/det_ppocrv4_server.yml",
       "--root", ROOT,
       "--weights", "/content/weights/ch_PP-OCRv4_det_server_train/best_accuracy",
       "--save-dir", f"{DRIVE}/experiments/det_v4_server",
       "--epochs", "50", "--batch-size", "8"]
if RESUME:
    cmd += ["--checkpoints", RESUME]

# config 생성이 조용히 실패하면 낡은 yml 로 50 epoch 을 돌리게 된다. 여기서 멈춘다.
assert sh(cmd) == 0, "config 생성 실패"

assert sh(["python", "tools/train.py", "-c", f"{ROOT}/configs/det_ppocrv4_server.yml"],
          cwd="/content/PaddleOCR") == 0, "학습 실패"

## 8. 평가 — M3 판정

두 층위로 나눠 본다(`docs/metrics_policy.md` 4장의 L2 / L3).

- **L2 모듈 KPI**: 검출 Hmean/recall — 학습이 먹혔는가
- **L3 시스템 KPI**: end-to-end Top-5 — **유일한 판정 기준**, 목표 ≥81%

In [ ]:
CKPT = f"{DRIVE}/experiments/det_v4_server/best_accuracy"
CFG  = f"{ROOT}/configs/det_ppocrv4_server.yml"
PDL  = "/content/PaddleOCR"

# L2 — 2.x 자체 평가 (val)
assert sh(["python", "tools/eval.py", "-c", CFG,
           "-o", f"Global.checkpoints={CKPT}"], cwd=PDL) == 0, "eval 실패"

# 추론 모델로 export (end-to-end 에 필요)
assert sh(["python", "tools/export_model.py", "-c", CFG,
           "-o", f"Global.pretrained_model={CKPT}",
           f"Global.save_inference_dir={DRIVE}/experiments/det_v4_server/inference"],
          cwd=PDL) == 0, "export 실패"

### L3 재측정에 대한 주의

`src/eval/end_to_end.py` 는 `paddleocr` 3.7 의 `TextDetection` 을 쓴다. 학습 결과는 2.x 로
export 한 추론 모델이므로, **검출기를 교체할 어댑터가 필요하다.** 이 노트북 범위 밖이며,
L2 개선이 확인된 뒤 로컬에서 어댑터를 붙여 재측정한다.

임시로는 학습 전후 **검출 recall** 만 비교해 방향을 확인하고,
`시스템 Top-5 ≈ 검출 recall × 85.6%(조건부 인식)` 로 추정한다.